In [24]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [25]:
# load the feature engineered dataset
df = pd.read_csv("../data/processed/featured_sales.csv")
df["date"] = pd.to_datetime(df["date"])

# checking shape to make sure dates are sorted sequentially for the split
df = df.sort_values(["item_name", "date"])
print(f"Data shape: {df.shape}")
df.head()  


Data shape: (10878, 11)


,date,item_name,quantity_clean,week,month,quarter,lag_1,lag_2,lag_4,rolling_mean_4,rolling_std_4
86,2024-01-30,Arizona Hard Green Tea 12 Pack (22 oz Cans),0.0,5,1,1,0.0,0.0,0.0,0.0,0.0
184,2024-02-06,Arizona Hard Green Tea 12 Pack (22 oz Cans),0.0,6,2,1,0.0,0.0,0.0,0.0,0.0
282,2024-02-13,Arizona Hard Green Tea 12 Pack (22 oz Cans),0.0,7,2,1,0.0,0.0,0.0,0.0,0.0
380,2024-02-20,Arizona Hard Green Tea 12 Pack (22 oz Cans),0.0,8,2,1,0.0,0.0,0.0,0.0,0.0
478,2024-02-27,Arizona Hard Green Tea 12 Pack (22 oz Cans),0.0,9,2,1,0.0,0.0,0.0,0.0,0.0


In [20]:
# 80 train 20 test split
split_date = df["date"].quantile(0.8)

train = df[df["date"] < split_date]
test  = df[df["date"] >= split_date]

print(f"Train set ranges from {train['date'].min().date()} to {train['date'].max().date()} ({train.shape[0]} rows)")
print(f"Test set ranges from  {test['date'].min().date()} to {test['date'].max().date()} ({test.shape[0]} rows)")


Train set ranges from 2024-01-30 to 2025-10-07 (8624 rows)
Test set ranges from  2025-10-14 to 2026-03-17 (2254 rows)


In [8]:
# exclude item_name to maintain a single global model across skus

features = ["lag_1", "lag_2", "lag_4", "rolling_mean_4", "rolling_std_4", "week", "month", "quarter"]
target = "quantity_clean"

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]


In [21]:
# predict that this week's demand exactly equals last week demand lag_1
y_pred_baseline = test["lag_1"]


In [22]:
# global linear regression model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# predict on the holdout test set
y_pred_lr = lr_model.predict(X_test)

# clip to 0 since neg beer values are returns and we dont need that in our training
y_pred_lr = np.clip(y_pred_lr, a_min=0, a_max=None)


In [26]:
# compare
def wmape(y_true, y_pred):
    """Calculate Weighted Mean Absolute Percentage Error (WMAPE)"""
    return np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)

def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    wmape_val = wmape(y_true, y_pred)
    
    print(f"--- {name} ---")
    print(f"MAE:   {mae:.4f}")
    print(f"RMSE:  {rmse:.4f}")
    print(f"WMAPE: {wmape_val:.4%}\n")

evaluate_model("Baseline (Lag 1)", y_test, y_pred_baseline)
evaluate_model("Standard Linear Regression", y_test, y_pred_lr)


--- Baseline (Lag 1) ---
MAE:   0.4818
RMSE:  1.1304
WMAPE: 99.5417%

--- Standard Linear Regression ---
MAE:   0.4897
RMSE:  0.9463
WMAPE: 101.1773%



(10878, 11)